# Nova Pay Fraud Detection — Steps 7 & 8
## Deployment (FastAPI + Docker) and Monitoring (Drift + Retraining)

**Where we are.** Steps 1–6 took raw data all the way to a trained, validated, explainable model saved as an artifact (`fraud_model.joblib`). The model now sits on disk doing nothing useful. Steps 7 and 8 are about *putting it to work and keeping it working*:

- **Step 7 — Deployment:** wrap the model in a small web service (a FastAPI `/score` endpoint) so any system can send it a transaction and get back a fraud score, then package that service in Docker so it runs identically anywhere.
- **Step 8 — Monitoring & Improvement:** watch live data for *drift* (the world changing under the model) and retrain on a schedule so accuracy doesn't quietly decay.

This notebook is written for a learner. We explain every concept in plain language first, show small runnable examples, and **actually test the service inside the notebook** so you can see it respond.

> A note on running this: the cells that *write files* (the API app, the Dockerfile) always work. The cell that *tests the API* uses FastAPI's built-in `TestClient`, which runs the app in-process — no server, no ports, works inside Jupyter.

## STEP 7 — DEPLOYMENT

### 📚 Concept 1: What does "deploy" actually mean?

Right now, using the model requires opening Python, loading the `.joblib` file, and calling `model.predict_proba(...)` by hand. That is fine for *you*, in a notebook. But Nova Pay's payment system is not a notebook — it is a live application that needs a fraud score for **every transaction, automatically, in milliseconds**.

**Deployment** means turning the model into a *service*: something that is always running and waiting, that other software can talk to over the network. The standard way to do this is an **API** (Application Programming Interface) — think of it as a waiter. The payment system (the customer) hands the waiter an order (the transaction details); the waiter takes it to the kitchen (the model); the kitchen cooks (scores it); the waiter brings back the dish (the fraud probability). The customer never goes into the kitchen.

We will build that waiter with **FastAPI**.

### 📚 Concept 2: What is FastAPI?

**FastAPI** is a Python library for building web APIs quickly. You write normal Python functions, decorate them to say "this function answers requests at this address," and FastAPI handles all the networking, request parsing, and response formatting for you.

Two key ideas:
- An **endpoint** is an address the service answers at, e.g. `/score` or `/health`. Each endpoint is just a Python function.
- An **HTTP method** says what *kind* of action it is. `GET` = "give me information" (used for a health check). `POST` = "here is some data, do something with it" (used to send a transaction for scoring).

### 📚 Concept 3: What is a "base model" — and why the word is confusing here

You asked specifically about the **base model** in relation to FastAPI. This is worth slowing down on, because **"base model" means two completely different things** depending on context, and mixing them up is a classic beginner trap:

**1. "Base model" in machine learning** = a simple, first-attempt model you compare everything else against. In Step 5 our *baseline (base) models* were Logistic Regression and Random Forest — the starting points we measured XGBoost and LightGBM against. This meaning has nothing to do with FastAPI.

**2. `BaseModel` in FastAPI** = a class from the **Pydantic** library that you inherit from to **describe the shape of your data**. When you write `class Transaction(BaseModel):` and list the fields, you are telling FastAPI: "a valid transaction must have these fields, with these types." FastAPI then *automatically*:
- parses incoming JSON into a Python object,
- **validates** it (rejecting bad input with a clear error), and
- documents it.

So in the sentence "the request body is a Pydantic `BaseModel`," `BaseModel` is the data-shape tool — **not** a machine-learning baseline. In this notebook, when we say *base model* near FastAPI we always mean the Pydantic `BaseModel`. When we mean the ML kind, we'll say *baseline model*.

### 7.1 — Write the FastAPI service

Now the service itself. Point the API to your actual Step-6 artifact. Read the comments — every piece maps to a concept we explained previously. We write it to a file (`service/app.py`) because that is how it will run in production; in the next cell we load that same app and test it.

### 📚 Why the request form is now small (and built automatically)

Earlier the model expected ~93–106 features, which would force a ~100-field `/score` request — impossible to hand-enter and brittle to call. The modelling notebook now saves a **lean artifact** whose `feature_cols` is the reduced set (≈12). 

The cell below builds the Pydantic `Transaction` model **dynamically from that list**, so the API form always matches the model exactly — no manual schema to maintain, and only a dozen fields to provide. (This also fixes two earlier bugs: the undefined `app_code` and `predict_proba(row)[1]`, which should be `[0,1]`.)

In [42]:
%pip install fastapi uvicorn joblib numpy pydantic -q

import os, joblib, numpy as np
os.makedirs("../service", exist_ok=True)

# ---------------------------------------------------------------------------
# Use the LEAN artifact saved by the modelling notebook (short feature list).
# ---------------------------------------------------------------------------
ARTIFACT_PATH = "../service/nova_pay_fraud_model_lean.joblib"

# ---------------------------------------------------------------------------
# Build the FastAPI app source DYNAMICALLY from FEATURES, so the request form
# always matches whatever the model expects — no hard-coded 100-field schema.
# ---------------------------------------------------------------------------
field_lines = []
for f in FEATURES:
    if "mismatch" in f or f.endswith("_flag"):                      # these are binary flags, so constrain to 0 or 1 
        field_lines.append(f"    {f}: int = Field(ge=0, le=1)")     # integers only, 0 or 1 
    else:
        field_lines.append(f"    {f}: float")                       # otherwise, just allow any float
fields_block = "\n".join(field_lines)                               # dynamically build the Pydantic BaseModel fields from the model's feature list, instead of hard-coding 100 fields in the source code

app_code = f''' 
from fastapi import FastAPI
from pydantic import BaseModel, Field
import joblib, numpy as np

artifact = joblib.load("{ARTIFACT_PATH}")                           # load the LEAN model artifact (short feature list)
model    = artifact["model"]                                        # the sklearn model object itself
FEATURES = artifact["feature_cols"]                                 # the model's own feature list (short form)
THRESH   = artifact["threshold"]                                    # the model's own threshold for classifying fraud

app = FastAPI(title="Nova Pay Fraud Scoring API", version="2.0")    # app title and version for the OpenAPI docs

# Pydantic BaseModel built from the model's OWN feature list (lean form).
class Transaction(BaseModel):
{fields_block}                                                      # dynamically built from the model's own feature list, so the request form always matches whatever the model expects

class ScoreResponse(BaseModel):
    fraud_probability: float                                        # probability of fraud (0.0-1.0)
    is_fraud: bool                                                  # whether the transaction is classified as fraud (True/False)
    threshold: float                                                # the threshold used for classification (0.0-1.0)

@app.get("/health")                                                 # health check endpoint for monitoring
def health(): 
    return {{"status": "ok", "n_features": len(FEATURES), "features": FEATURES}}    # Return health status and feature information

@app.post("/score", response_model=ScoreResponse)                                   # score endpoint for scoring transactions
def score(txn: Transaction):
    row = np.array([[getattr(txn, f) for f in FEATURES]])                           # build a 2D array (1 row, n_features columns) from the Transaction object, in the order of FEATURES
    prob = float(model.predict_proba(row)[0, 1])   # [0,1] = P(fraud) for row 0     # define the probability of fraud for the transaction using the model's predict_proba method
    return ScoreResponse(fraud_probability=round(prob, 4),                          # round to 4 decimal places
                         is_fraud=prob >= THRESH, threshold=THRESH)                 # return the scoring results
'''

with open("../service/app.py", "w") as f:                                              # write the dynamically generated FastAPI app code to service/app.py
    f.write(app_code)
print("✓ Wrote ../service/app.py with a", len(FEATURES), "field form (was ~100).")

Note: you may need to restart the kernel to use updated packages.
✓ Wrote ../service/app.py with a 6 field form (was ~100).



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### 7.2 — TEST the service (inside the notebook)

This is the part that proves it works. FastAPI ships with a `TestClient` that runs the app *in memory* — no server to start, no port to open — so it works cleanly inside Jupyter. We will:

1. hit `/health` (a GET) to confirm the service is up,
2. send a **legit-looking** transaction to `/score` and expect a low probability,
3. send a **fraud-looking** transaction and expect a high probability,
4. send a **broken** request (missing fields) and watch the `BaseModel` reject it with HTTP 422.

In [43]:
%pip install httpx -q

import warnings; warnings.filterwarnings("ignore")
import importlib.util, numpy as np

spec = importlib.util.spec_from_file_location("nova_app", "../service/app.py")                 # load the FastAPI app from the dynamically generated service/app.py
nova_app = importlib.util.module_from_spec(spec); spec.loader.exec_module(nova_app)         # execute the module to initialize the FastAPI app
from fastapi.testclient import TestClient                                                   # create a TestClient for testing the FastAPI app
client = TestClient(nova_app.app) 

FEATURES = nova_app.FEATURES

# 1) Health check
print("1) HEALTH:", client.get("/health").json())

# Build legit / fraud sample requests for WHATEVER features the model expects.
def make_request(direction):
    sign = -1.0 if direction == "legit" else 1.0
    req = {}
    for f in FEATURES:
        if "mismatch" in f or f.endswith("_flag"):                  # binary flags: 0=legit, 1=fraud
            req[f] = 0 if direction == "legit" else 1
        elif "trust" in f:                                          # high trust = legit
            req[f] = 1.2 if direction == "legit" else -1.5
        else:                                                       # risk/velocity/amount: low=legit
            req[f] = sign * 1.5
    return req

print("2) LEGIT-LIKE :", client.post("/score", json=make_request("legit")).json())
print("3) FRAUD-LIKE :", client.post("/score", json=make_request("fraud")).json())

# 4) Broken request: missing fields -> BaseModel rejects with 422
print("4) BROKEN REQUEST -> HTTP",
      client.post("/score", json={FEATURES[0]: 1.0}).status_code,
      "(422 = validation rejected it, as intended)")

Note: you may need to restart the kernel to use updated packages.
1) HEALTH: {'status': 'ok', 'n_features': 6, 'features': ['txn_velocity_1h', 'txn_velocity_24h', 'ip_risk_score', 'device_trust_score', 'country_location_mismatch', 'amount_usd']}
2) LEGIT-LIKE : {'fraud_probability': 0.0167, 'is_fraud': False, 'threshold': 0.5}
3) FRAUD-LIKE : {'fraud_probability': 0.9583, 'is_fraud': True, 'threshold': 0.5}
4) BROKEN REQUEST -> HTTP 422 (422 = validation rejected it, as intended)



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


**What you should see:** the legit transaction returns a low `fraud_probability` with `is_fraud: false`; the fraud-looking one returns a high probability with `is_fraud: true`; and the broken request comes back as **HTTP 422 Unprocessable Entity**. That 422 is the Pydantic `BaseModel` doing its job — the malformed transaction never reached the model. This is the single most important safety property of using a `BaseModel` as your endpoint's front door.

### 📚 Concept 4: How this runs "for real"

In production you don't use `TestClient`. You run the service with an ASGI server called **uvicorn**:

```bash
uvicorn service.app:app --host 0.0.0.0 --port 8000
```

Then the payment system sends a real HTTP request:

```bash
curl -X POST http://localhost:8000/score \
     -H "Content-Type: application/json" \
     -d '{"txn_velocity_1h":2.5,"txn_velocity_24h":2.2,"ip_risk_score":2.0,
          "device_trust_score":-1.5,"country_location_mismatch":1,"amount_usd":1.0}'
```

FastAPI also auto-generates interactive documentation at `http://localhost:8000/docs` — you get a clickable test page for free, built from your `BaseModel` definitions.

### 7.3 — Containerise with Docker

### 📚 Concept 5: What is Docker, and why container-ise?

Your service works *on your machine*. But "it works on my machine" is famously where deployment goes wrong — the server might have a different Python version, missing libraries, or different settings. **Docker** solves this by packaging your service **and everything it needs** (Python, libraries, the model file) into a single **image**. That image runs identically on your laptop, a colleague's machine, or a cloud server. A running copy of an image is called a **container**.

The recipe for an image is a **Dockerfile**. We write one below. (We can't run Docker inside this notebook, but the file is exactly what you'd use.)

In [44]:
dockerfile = '''# Start from a small official Python image
FROM python:3.11-slim

# Set the working directory inside the container
WORKDIR /app

# Install dependencies first (this layer is cached for faster rebuilds)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy the service code and the model artifact into the image
COPY ../service/ ./service/

# Document the port the service listens on
EXPOSE 8000

# The command that runs when the container starts
CMD ["uvicorn", "service.app:app", "--host", "0.0.0.0", "--port", "8000"]
'''
with open("../service/Dockerfile", "w") as f:
    f.write(dockerfile)

requirements = "fastapi\nuvicorn\npydantic\nscikit-learn\njoblib\nnumpy\n"
with open("../service/requirements.txt", "w") as f:
    f.write(requirements)

print("Wrote Dockerfile and requirements.txt\n")
print(dockerfile)

Wrote Dockerfile and requirements.txt

# Start from a small official Python image
FROM python:3.11-slim

# Set the working directory inside the container
WORKDIR /app

# Install dependencies first (this layer is cached for faster rebuilds)
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy the service code and the model artifact into the image
COPY ../service/ ./service/

# Document the port the service listens on
EXPOSE 8000

# The command that runs when the container starts
CMD ["uvicorn", "service.app:app", "--host", "0.0.0.0", "--port", "8000"]



# To build and run it (in a terminal, not the notebook):

```bash
docker build -t nova-fraud-api .          # build the image from the Dockerfile
docker run -p 8000:8000 nova-fraud-api    # run a container, mapping port 8000
```

# After that, the `/score` endpoint is reachable at `http://localhost:8000/score` exactly as before — but now it runs the same way on any machine with Docker. **That portability is the whole point of Step 7.**

## STEP 8 — MONITORING & IMPROVEMENT

### 📚 Concept 6: Why a deployed model needs monitoring

A model learns the patterns present *in the data it was trained on*. But fraud is a moving target: fraudsters change tactics, new payment channels appear, customer behaviour shifts. Over time, the live data starts to look different from the training data. This is called **drift**, and it silently erodes accuracy — the model keeps answering confidently while slowly becoming wrong.

Two kinds of drift to know:
- **Data drift:** the *inputs* change distribution (e.g. average transaction velocity creeps up as automated fraud rises).
- **Concept drift:** the *relationship* between inputs and fraud changes (e.g. a feature that used to signal fraud no longer does).

Monitoring means continuously comparing **recent live data** against the **training (reference) data** and raising an alert when they diverge enough to matter.

### 📚 Concept 7: Measuring drift with PSI (a simple, transparent start)

The workflow specifies **Evidently** for drift detection (shown as a reference at the end). But to *understand* drift, it helps to compute it by hand once. A common, easy metric is the **Population Stability Index (PSI)**, which compares two distributions of a feature:

- **PSI < 0.1** → no meaningful drift
- **0.1 ≤ PSI < 0.2** → moderate drift, worth watching
- **PSI ≥ 0.2** → significant drift, investigate / consider retraining

We compute PSI for a feature where nothing changed, and one where the distribution shifted, so you can see the number react.

In [45]:
import numpy as np

def population_stability_index(reference, live, bins=10):
    """Compare a 'reference' (training) feature against 'live' data."""
    # Use the reference quantiles as bin edges
    edges = np.quantile(reference, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    ref_pct  = np.histogram(reference, bins=edges)[0] / len(reference)
    live_pct = np.histogram(live,      bins=edges)[0] / len(live)
    # Avoid division by zero
    ref_pct  = np.clip(ref_pct, 1e-4, None)
    live_pct = np.clip(live_pct, 1e-4, None)
    return float(np.sum((live_pct - ref_pct) * np.log(live_pct / ref_pct)))

rng = np.random.RandomState(0)
reference     = rng.normal(0.0, 1.0, 5000)   # training distribution of a feature
live_stable   = rng.normal(0.0, 1.0, 5000)   # later data, unchanged
live_drifted  = rng.normal(0.6, 1.2, 5000)   # later data, shifted & wider

print(f"PSI (no drift):   {population_stability_index(reference, live_stable):.3f}  -> stable")
print(f"PSI (with drift): {population_stability_index(reference, live_drifted):.3f}  -> investigate")

PSI (no drift):   0.001  -> stable
PSI (with drift): 0.327  -> investigate


You should see the stable comparison near **0.00** and the drifted one **above 0.2**. In production you would compute PSI (or Evidently's equivalent) for each feature on a schedule — say weekly — and alert when any feature crosses the threshold. That alert is your early warning that the model may be going stale.

### 📚 Concept 8: The quarterly retraining playbook

Monitoring tells you *when* something is wrong; retraining is *how you fix it*. The workflow calls for a **quarterly retraining playbook** — a documented, repeatable routine rather than an ad-hoc scramble. A sensible playbook:

1. **Collect** the last quarter's transactions, now with confirmed fraud/not-fraud labels from investigators.
2. **Check drift** (PSI / Evidently) to confirm whether the world has actually changed.
3. **Re-run Steps 2–6** of the pipeline on the refreshed data: clean, engineer, train, tune, validate.
4. **Compare** the new model against the current production model on a fresh holdout — *only promote it if it is genuinely better*.
5. **Shadow-deploy** the candidate alongside production (score both, act on neither new one) to confirm live behaviour.
6. **Promote** the new model and archive the old one (so you can roll back).
7. **Document** what changed and why.

The pseudo-code below sketches the decision logic — it does not run a full retrain, it shows the *shape* of the routine.

In [46]:
# Pseudo-code sketch of the retraining decision (illustrative, not a live run).
def quarterly_retraining_playbook(new_data_available=True, max_feature_psi=0.27,
                                  current_pr_auc=0.92, candidate_pr_auc=0.94):
    print("Quarterly retraining check")
    print("-" * 40)
    if not new_data_available:
        print("No new labelled data yet — skip this cycle."); return

    drift_flag = max_feature_psi >= 0.2
    print(f"Max feature PSI: {max_feature_psi:.2f} -> "
          f"{'DRIFT detected' if drift_flag else 'no significant drift'}")

    # Always retrain on fresh labels, but only PROMOTE if better
    print(f"Current model PR-AUC:   {current_pr_auc:.3f}")
    print(f"Candidate model PR-AUC: {candidate_pr_auc:.3f}")
    if candidate_pr_auc > current_pr_auc:
        print("DECISION: candidate is better -> shadow-deploy, then promote.")
    else:
        print("DECISION: candidate not better -> keep current model, investigate.")

quarterly_retraining_playbook()

Quarterly retraining check
----------------------------------------
Max feature PSI: 0.27 -> DRIFT detected
Current model PR-AUC:   0.920
Candidate model PR-AUC: 0.940
DECISION: candidate is better -> shadow-deploy, then promote.


### 📚 Concept 9: Evidently (the tool the workflow specifies)

`Evidently` automates what we did by hand with PSI, across all features at once, and produces a shareable report/dashboard. The exact API changes between Evidently versions, so treat the snippet below as a **reference pattern** rather than copy-paste code — check the version installed in your environment. Conceptually it does exactly what our PSI function did: compare a **reference** dataset to **current** data and flag drift.

```python
# Reference pattern (API varies by Evidently version):
# from evidently.report import Report
# from evidently.metric_preset import DataDriftPreset
#
# report = Report(metrics=[DataDriftPreset()])
# report.run(reference_data=training_df, current_data=live_df)
# report.save_html("drift_report.html")   # open in a browser
```

The takeaway is the *routine*, not the library: compare recent live data to your training baseline regularly, alert on drift, and feed that signal into the quarterly retraining decision.

## SUMMARY — Steps 7 & 8

**Step 7 (Deployment).** We turned the saved model into a live **FastAPI** service with a `/score` endpoint and a `/health` check, used a Pydantic **`BaseModel`** as the validating front door (the *FastAPI* sense of "base model", distinct from an ML *baseline* model), **tested it in-notebook** with `TestClient` — confirming sensible scores and a 422 rejection of bad input — and wrote a **Dockerfile** so the service runs identically anywhere.

**Step 8 (Monitoring & Improvement).** We explained **drift** (data vs. concept), measured it transparently with **PSI**, sketched the **quarterly retraining playbook** (retrain on fresh labels, promote only if better, shadow-deploy, keep rollback), and pointed to **Evidently** as the production tool that automates drift reporting.

**The big picture:** Steps 1–6 build a good model; Steps 7–8 make it *useful and durable* — served reliably, and kept honest as the world changes.